# FL-IDS — Colab main notebook
ABAC-secured, Byzantine-resilient federated IoT IDS (see `Architecture_README.md`).

Run the cells top to bottom. CPU runtime is enough. Each stage adds cells at the end.

## 1. Mount Google Drive (datasets + results live here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repository
The repo is private: paste a GitHub **fine-grained token (read-only, this repo only)** when asked.
The token is used once and removed from the git remote afterwards; it is never saved in the notebook.

In [ ]:
import os, getpass, subprocess
REPO = 'Devendrabhuarya/Phase1Claude'
REPO_DIR = '/content/fl-ids'
if not os.path.isdir(REPO_DIR):
    token = getpass.getpass('GitHub token: ')
    subprocess.run(['git', 'clone', '-q', f'https://{token}@github.com/{REPO}.git', REPO_DIR], check=True)
    # drop the token from the stored remote URL
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://github.com/{REPO}.git'], check=True)
    del token
else:
    print('Repo already cloned. To update: run `git pull` with a token, or delete', REPO_DIR, 'and re-run this cell.')
%cd /content/fl-ids
!git log --oneline -5

## 3. Install missing dependencies
Colab already ships torch, numpy, pandas, scikit-learn, matplotlib and pyyaml; only PyCasbin is added.

In [ ]:
!pip install -q pycasbin==2.8.0
import torch, numpy, pandas, sklearn, casbin
print('torch', torch.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| sklearn', sklearn.__version__)

## 4. Run the unit tests

In [ ]:
!python -m pytest -q

## 5. Load the config and check the data folder

In [ ]:
import sys, os
sys.path.insert(0, '/content/fl-ids')
from src.utils.config import load_config
from src.utils.seed import set_seed

cfg = load_config('ciciot2023')
set_seed(cfg['seed'])
d = cfg['paths']['data_dir']
print(d, 'entries:', len(os.listdir(d)) if os.path.isdir(d) else 'MISSING')
os.makedirs(cfg['paths']['results_dir'], exist_ok=True)
print('results ->', cfg['paths']['results_dir'])

## Stage 1 — CICIoT2023 loading, label mapping, splits
**1a.** Quick check on the first 3 files (a few minutes, no cache written).
If a label is not mapped, an error lists it: send it to me and I will add the keyword.

In [ ]:
!python experiments/stage1_data.py --inspect-only --max-files 3
!python experiments/stage1_data.py --max-files 3 --no-cache

**1b.** Full run over all 63 files (slow, ~20-30 min, done once).
The class-capped sample is cached in `fl-ids-results/cache/` and reused by later stages.

In [ ]:
!python experiments/stage1_data.py

**1c.** Use the prepared data in Python (loads from the cache).

In [ ]:
from src.data.preprocess import prepare_dataset
bundle = prepare_dataset(cfg)
print(bundle.X_pool.shape, bundle.X_test.shape, bundle.X_root.shape)